# Bài toán

> **Thu thập dữ liệu báo Việt Nam (VnExpress - Demo)**

Mục tiêu:

- Hiểu được quy trình thu thập dữ liệu từ các trang báo Việt Nam
- Thu thập dữ liệu (bài báo) từ các trang báo Việt Nam để làm dữ liệu cho các bước xử lý sau

Đầu ra:

-  Tập file JSON chứa các bài bài báo có các trường dữ liệu:

    - `url`: link dẫn đến bài báo
    - `title`: tiêu đề bài báo
    - `description`: tóm tắt bài báo
    - `content`: nội dung bài báo
    - `metadata`: trường dữ liệu bổ sung

        - `cat`: thể loại bài báo
        - `subcat`: thể loại con của bài báo
        - `published_date`: thời gian xuất bản
        - `author`: người viết
    
- Ví dụ về một bài báo:

    ```
    {
        "url": "https://vnexpress.net/chinh-phu-ban-hanh-nghi-dinh-moi-ve-gia-dat-4763835.html",
        "title": "Chính phủ ban hành nghị định mới về giá đất",
        "description": "Chính phủ hôm nay ban hành Nghị định 71, trong đó quy ...",
        "content": "Nghị định này có hiệu lực khi Luật Đất đai 2024 được thi hành ...",
        "metadata": {
            "cat": "Bất động sản",
            "subcat": "Chính sách",
            "published_date": 1719575647,
            "author": "Anh Tú"
        }
    },
    ```

# Các bước tiến hành

## Chuẩn bị các thư viện cần thiết

In [ ]:
# Suitable for Google Colab, for local please follow the external instructions and ignore this line
# and follows https://docs.google.com/document/d/14jK9d6KHJYX0b-gFAVqAghUxT7OLAM0nP2IovL7_Rjs/edit?usp=sharing
!apt install -qq chromium-chromedriver

The following additional packages will be installed:
  apparmor chromium-browser libfuse3-3 liblzo2-2 libudev1 snapd squashfs-tools systemd-hwe-hwdb
  udev
Suggested packages:
  apparmor-profiles-extra apparmor-utils fuse3 zenity | kdialog
The following NEW packages will be installed:
  apparmor chromium-browser chromium-chromedriver libfuse3-3 liblzo2-2 snapd squashfs-tools
  systemd-hwe-hwdb udev
The following packages will be upgraded:
  libudev1
1 upgraded, 9 newly installed, 0 to remove and 44 not upgraded.
Need to get 28.5 MB of archives.
After this operation, 118 MB of additional disk space will be used.
Preconfiguring packages ...
Selecting previously unselected package apparmor.
(Reading database ... 123588 files and directories currently installed.)
Preparing to unpack .../apparmor_3.0.4-2ubuntu2.3_amd64.deb ...
Unpacking apparmor (3.0.4-2ubuntu2.3) ...
Selecting previously unselected package liblzo2-2:amd64.
Preparing to unpack .../liblzo2-2_2.10-2build3_amd64.deb ...
Unpack

In [ ]:
# Install selenium
!pip install -qq selenium

# Tạo thư mục để chứa data
!mkdir -p data

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 475.7/475.7 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.8 MB/s eta 0:00:00


In [1]:
# selenium import
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.common.exceptions import StaleElementReferenceException, NoSuchElementException

# other imports
import os
import json

In [2]:
# selenium setups
## https://www.tutorialspoint.com/selenium/selenium_webdriver_chrome_webdriver_options.htm
chromedriver_path = "D:\DriverResources\chromedriver-win64\chromedriver.exe"

chrome_options = webdriver.ChromeOptions()

chrome_options.add_argument('--headless') # must options for Google Colab
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument("--disable-extensions")
chrome_options.add_argument("--disable-gpu")

# Set the path to the chromedriver
chrome_options.add_argument(f"webdriver.chrome.driver={chromedriver_path}")

In [3]:
MAGAZINE_NAME = "vnexpress"
HOME_PAGE = "https://vnexpress.net/"

## Thu thập dữ liệu

> **Các bước thu thập bài báo**


1. News categories: Thu thập tất cả các thể loại báo của website
2. News urls: Thu thập một số đường dẫn dựa vài từng thể loại báo của website đó
3. News articles: Thu thập và xử lý từng bài báo dựa vào đường dẫn của bước trước

### Thu thập thể loại bài báo của website: Crawling categories

> **Các bước thu thập**

1. Vào trang chủ của báo
2. Thu thập các thể loại ở mục menu


Vào trang chủ

In [18]:
driver = webdriver.Chrome(options=chrome_options)
# Vào trang web chính, mặc định phải chờ toàn bộ trang webload mới xong
driver.get(HOME_PAGE)

In [5]:
from selenium.common.exceptions import WebDriverException
try:
    # Thử lấy tiêu đề của trang hiện tại
    current_title = driver.title
    print(f"Browser title is: {current_title}")
except WebDriverException as e:
    print(f"Connection issue: {e}")

Browser title is: Báo VnExpress - Báo tiếng Việt nhiều người xem nhất


Chọn menu buttons

In [7]:
# Chọn menu buttons

## Chọn element theo class_name == "all-menu"
all_menu = driver.find_element(by=By.CLASS_NAME, value="all-menu")

## Click vào menu
all_menu.click() #

Thu thập hết thể loại:

* Cách chọn elements ở web trong selenium: https://selenium-python.readthedocs.io/locating-elements.html

In [8]:
# Lấy hết tất cả thể loại từ đây
cats = []

# Chọn row-menu bằng class_name
row_menu = driver.find_element(by=By.CLASS_NAME, value="row-menu")

# Ta thấy mỗi thể loại ở trong element là cat-menu element --> ta lấy hết elements bằng dùng find_elements; find_element -> chọn element đầu tiên match
cat_menus = row_menu.find_elements(by=By.CLASS_NAME, value="cat-menu")

for cat_menu in cat_menus: # Loop through cat menus to main cat and corresponding links
    # Lấy element đầu tiên
    cat = cat_menu.find_element(by=By.TAG_NAME, value="a").get_attribute("title").strip()
    href = cat_menu.find_element(by=By.TAG_NAME, value="a").get_attribute("href").strip()
    # Lưu vào cats list bao gồm tên và url
    cats.append({"cat_name": cat, "url": href})


In [9]:
cats, len(cats)

([{'cat_name': 'Thời sự', 'url': 'https://vnexpress.net/thoi-su'},
  {'cat_name': 'Góc nhìn', 'url': 'https://vnexpress.net/goc-nhin'},
  {'cat_name': 'Thế giới', 'url': 'https://vnexpress.net/the-gioi'},
  {'cat_name': 'Video', 'url': 'https://video.vnexpress.net/'},
  {'cat_name': 'Podcasts', 'url': 'https://vnexpress.net/podcast'},
  {'cat_name': 'Kinh doanh', 'url': 'https://vnexpress.net/kinh-doanh'},
  {'cat_name': 'Bất động sản', 'url': 'https://vnexpress.net/bat-dong-san'},
  {'cat_name': 'Khoa học', 'url': 'https://vnexpress.net/khoa-hoc'},
  {'cat_name': 'Giải trí', 'url': 'https://vnexpress.net/giai-tri'},
  {'cat_name': 'Thể thao', 'url': 'https://vnexpress.net/the-thao'},
  {'cat_name': 'Pháp luật', 'url': 'https://vnexpress.net/phap-luat'},
  {'cat_name': 'Giáo dục', 'url': 'https://vnexpress.net/giao-duc'},
  {'cat_name': 'Sức khỏe', 'url': 'https://vnexpress.net/suc-khoe'},
  {'cat_name': 'Đời sống', 'url': 'https://vnexpress.net/doi-song'},
  {'cat_name': 'Du lịch', 'u

In [10]:
# Đóng driver sau khi dùng xong
driver.close()

### Thu thập một số đường dẫn dựa vài từng thể loại báo của website đó: News urls


> **Cách thu thập**

Từ những thể loại và đường link tương ứng, ta lần lượt vào từng thể loại và lấy đường dẫn của các bài báo trong mục đó của trang web đó.



#### Cài đặt thông số

In [9]:
# Đặt số lượng đường dẫn cần lấy trong mỗi thể loại báo
NUM_ARTICLES_PER_CAT = 500 # có thể tăng lên

# Đường dẫn lưu data của vnexpress
DATA_URL_FILE = "data/vnexpress_url.json"

# Một số thể loại không quan tâm
EXCLUDING_CATEGORIES = ["Video", "Podcasts", "Góc nhìn", "Tâm sự", "Thư giãn", "Ý kiến"]

# Bổ sung cài đặt chromedriver
## Ta đặt load stategy ở đây là normal: https://www.selenium.dev/documentation/webdriver/drivers/options/
chrome_options.page_load_strategy = "normal"

#### Chạy thử nghiệm

In [12]:
driver = webdriver.Chrome(options=chrome_options)

In [10]:
# Vào thử một url
# driver.get(cats[0]['url'])
driver.get('https://vnexpress.net/giai-tri')

In [11]:
# Get all item-news class item
title_news = driver.find_elements(by=By.CLASS_NAME, value="title-news")

In [12]:
# Lấy đường dẫn
url_new = title_news[0].find_element(by=By.TAG_NAME, value="a").get_attribute("href")
url_new

'https://vnexpress.net/bang-kieu-tang-con-trai-xe-hoi-4785643.html'

In [13]:
# Chỉ lấy links bắt đầu giống HOME_PAGE
url_new.startswith(HOME_PAGE)

True

In [14]:
# Tìm next page
driver.find_element(by=By.CLASS_NAME, value="next-page").get_attribute("href")

'https://vnexpress.net/giai-tri-p2'

In [18]:
driver.close()

#### Chạy thật


In [21]:
driver = webdriver.Chrome(options=chrome_options)

In [59]:
# Global variables for filtering deduplicating urls
crawled_urls = set()

def crawl_each_category_url(driver, category_url):
    """
    Functions cho lấy urls cho từng category sau khi thử nghiệm
    """
    all_urls = set()
    url = category_url

    # Limit the number of NUM_ARTICLES_PER_CAT

    while len(all_urls) < NUM_ARTICLES_PER_CAT:
        driver.get(url)
        title_news = driver.find_elements(by=By.CLASS_NAME, value="title-news")
        for title in title_news:
            try:
                url_new = title.find_element(by=By.TAG_NAME, value="a").get_attribute("href")
                if url_new.startswith(HOME_PAGE) and url_new not in crawled_urls: #avoid ads, different sites news
                    all_urls.add(url_new)
                    crawled_urls.add(url_new) # avoid dedup url

            # To see if there is bug
            except StaleElementReferenceException:
                continue
            except NoSuchElementException:
                print(f"NoSuchElementException at {url}")
                continue

        url = driver.find_element(by=By.CLASS_NAME, value="next-page").get_attribute("href")

    return all_urls

In [24]:
saved_cats = {}

# Thu thập cho từng thể loại
for cat in cats:
    cat_name = cat["cat_name"]
    url = cat["url"]
    if cat_name not in EXCLUDING_CATEGORIES:
        print(f"You are at {cat}.")
        urls = crawl_each_category_url(driver, url)
        saved_cats[cat_name] = list(urls)

with open(DATA_URL_FILE, "w", encoding="utf-8") as fOut:
    json.dump(saved_cats, fOut, ensure_ascii=False, indent=4)

driver.close()

In [25]:
len(crawled_urls)

7172

### Thu thập và xử lý từng bài báo dựa vào đường dẫn của bước trước: News articles


> **Cách thu thập**

Từ đường dẫn ở trong phần trước, ta lần lượt vào từng đường link đó và thu thập thông tin về bài báo.

#### Cài đặt thông số

In [6]:
# Filepath cho cái trước
FILE_URL_PATH = "data/vnexpress_url.json"

# Đặt limit số articles lấy từ mỗi thể loại
MAX_ARTICLES_PER_CAT = None # Nếu set = None thì là tất cả các urls ở file trước

# Data output, mỗi thể loại là 1 file json chứa articles
DATA_FOLDER_OUTPUT = "data/vnexpress"
#
os.makedirs(DATA_FOLDER_OUTPUT, exist_ok=True)

# Để loading strategy về eager load nhanh, không quan tâm ảnh
chrome_options.page_load_strategy = "eager"

In [7]:
# Đọc file url
with open(FILE_URL_PATH, "r", encoding="utf-8") as fIn:
    url_data = json.load(fIn)

len(url_data)

14

In [10]:
url_data.keys()

dict_keys(['Thời sự', 'Thế giới', 'Kinh doanh', 'Bất động sản', 'Khoa học', 'Giải trí', 'Thể thao', 'Pháp luật', 'Giáo dục', 'Sức khỏe', 'Đời sống', 'Du lịch', 'Số hóa', 'Xe'])

#### Chạy thử nghiệm

In [28]:
driver = webdriver.Chrome(options=chrome_options)

# Một số url để thử nghiệm
SAMPLE_ARTICLE_URLS = [
    "https://vnexpress.net/lao-tuyen-bo-quoc-tang-tuong-niem-tong-bi-thu-nguyen-phu-trong-4772895.html",
    "https://vnexpress.net/ty-le-hoc-sinh-do-tot-nghiep-cao-nhat-10-nam-4772931.html",
    "https://vnexpress.net/tra-luong-cho-nguoi-gia-4772632.html",
    "https://vnexpress.net/cuoc-song-do-thi-mien-nam-hon-100-nam-truoc-4772382.html",
    "https://vnexpress.net/piastri-lan-dau-thang-chang-f1-4772734.html", # table
    "https://vnexpress.net/10-ung-vien-gianh-qua-bong-vang-2024-4772973.html", # slideshow,
    "https://vnexpress.net/toi-kho-chiu-khi-nhieu-nguoi-dung-xe-khi-con-ba-giay-den-xanh-4772885.html", # author,
    "https://vnexpress.net/khong-vo-gao-truoc-khi-nau-co-an-toan-4772624.html",
    "https://vnexpress.net/do-ban-thay-con-cong-khac-biet-trong-10-giay-4770515.html",
]

# chọn url
SAMPLE_ARTICLE_URL = SAMPLE_ARTICLE_URLS[1]

In [39]:
SAMPLE_ARTICLE_URL = "https://vnexpress.net/the-kho-cua-ong-trump-khi-ba-harris-troi-day-4785128.html"

In [40]:
driver.get(SAMPLE_ARTICLE_URL)

In [41]:
# Tìm kiếm title
driver.find_element(by=By.CSS_SELECTOR, value="h1.title-detail").text

'Thế khó của ông Trump khi bà Harris trỗi dậy'

In [42]:
# Tìm kiếm description
driver.find_element(by=By.CLASS_NAME, value="description").text

'Sự trỗi dậy của bà Harris khiến ông Trump mất đi sự chú ý từ công chúng, trong khi cựu tổng thống chưa tìm ra cách hiệu quả để ứng phó.'

In [43]:
# Thu thập thể loại
lis_cat = driver.find_element(by=By.CSS_SELECTOR, value="ul.breadcrumb").find_elements(by=By.TAG_NAME, value="li")
main_cat = lis_cat[0].text if len(lis_cat) > 0 else None
sub_cat = lis_cat[1].text if len(lis_cat) > 1 else None
main_cat, sub_cat

('Thế giới', 'Bầu cử tổng thống Mỹ 2024')

In [44]:
# Thu thập ngày
publish_date = driver.find_element(by=By.CSS_SELECTOR, value='[itemprop="datePublished"]').get_attribute("content").strip()
publish_date

'2024-08-25T19:00:00+07:00'

In [45]:
# Tìm kiếm contents
article = driver.find_element(by=By.CSS_SELECTOR, value="article.fck_detail")
children = article.find_elements(by=By.XPATH, value="./*")

In [53]:
# Thu thập contents và author
contents = []
author = "Unknown"
is_slide_show = False

for idx, child in enumerate(children):
    text = child.text.strip()
    # right align
    if child.tag_name == "p" and ("right" in child.get_attribute("align") or "right" in child.get_attribute("style")) and idx >= len(children) - 3: # last three, align right --> author
        author = text
    elif child.tag_name == "p" and child.get_attribute("class") == "Normal": # paragraph
        # If center
        if len(text):
            if ("center" in child.get_attribute("align") or "center" in child.get_attribute("style")):
                contents.append(f"[{text}]")
            else:
                contents.append(text)
    elif child.tag_name == "figure" :
        ## If length > 150  --> not a caption, it's next description
        if len(text):
            if len(text) <= 150:
                contents.append(f"[{text}]")
            else:
                contents.append(text)
    elif child.tag_name == "div" and "item_slide_show" in child.get_attribute("class"):
        is_slide_show = True # slideshow
        if len(text):
            if len(text) <= 100:
                contents.append(f"[{text}]")
            else:
                contents.append(text)

    elif child.tag_name == "table": # Do nothing rightnow
        pass

if is_slide_show:
    author = text

if author == "Unknown":
    try:
        author = driver.find_element(by=By.XPATH, value="//*[contains(@class, 'author')]").text
    except:
        pass

In [54]:
contents

['Đảng Dân chủ ngày 22/8 kết thúc đại hội toàn quốc dài 4 ngày, với bà Kamala Harris là tiêu điểm. Phó tổng thống Mỹ phát biểu nhận đề cử và làm nên lịch sử, trở thành nữ ứng viên da màu, nữ ứng viên gốc Á đầu tiên đại diện một chính đảng chạy đua vào Nhà Trắng.',
 'Việc bà Harris thay Tổng thống Joe Biden tranh cử đã giúp hồi sinh nỗ lực của đảng Dân chủ, vốn ảm đạm sau khi ông Biden gây thất vọng trong cuộc tranh luận trực tiếp với đối thủ đảng Cộng hòa Donald Trump cuối tháng 6. Bà Harris trong tháng qua gây quỹ vượt ông Trump, thu hẹp khoảng cách hoặc dẫn trước đối thủ trong nhiều cuộc thăm dò tại các bang chiến trường.',
 'Trong khi đó, ông Trump dường như chưa tìm ra phương án đối phó hiệu quả, dù cựu tổng thống từng nhiều lần khiến các đối thủ khác gặp bất lợi. Để thu hút sự chú ý trở lại của truyền thông, ông đồng ý trả lời phỏng vấn tỷ phú Elon Musk, thậm chí là tung ra hàng loạt lời công kích cá nhân cũng như những bình luận cường điệu nhắm vào bà Harris.',
 '"Sự phẫn nộ và n

In [69]:
author

'Dương Tâm'

In [70]:
driver.close()

#### Chạy thật


In [14]:
def get_content_metadata(driver, article_url):

    """
    Extracts and returns metadata and content from a given article URL.

    :param driver: Selenium WebDriver instance.
    :param article_url: URL of the article to extract data from.
    :return: Dictionary containing article metadata and content.
    """

    # Get to current article
    driver.get(article_url)

    # Thu thập title
    title = driver.find_element(by=By.CSS_SELECTOR, value="h1.title-detail").text.strip()

    # Thu thập description
    description = driver.find_element(by=By.CLASS_NAME, value="description").text.strip()

    # Thu thập thể loại
    lis_cat = driver.find_element(by=By.CSS_SELECTOR, value="ul.breadcrumb").find_elements(by=By.TAG_NAME, value="li")
    main_cat = lis_cat[0].text if len(lis_cat) > 0 else None
    sub_cat = lis_cat[1].text if len(lis_cat) > 1 else None

    # Thu thập published date
    publish_date = driver.find_element(by=By.CSS_SELECTOR, value='[itemprop="datePublished"]').get_attribute("content").strip()

    # Thu thập content bài báo
    # Locate phần viết content
    article = driver.find_element(by=By.CSS_SELECTOR, value="article.fck_detail")
    # Lấy hết các đầu mục con của bài báo
    children = article.find_elements(by=By.XPATH, value="./*")

    contents = []
    author = "Unknown"

    # Check có phải dạng slide show hay không
    is_slide_show = False
    for idx, child in enumerate(children):
        text = child.text.strip()
        # Nếu mà element right align --> có thể là tác giả
        if child.tag_name == "p" and ("right" in child.get_attribute("align") or "right" in child.get_attribute("style")) and idx >= len(children) - 3: # last three, align right --> author
            author = text
        elif child.tag_name == "p" and child.get_attribute("class") == "Normal": # paragraph
            # If center
            if len(text):
                if ("center" in child.get_attribute("align") or "center" in child.get_attribute("style")):
                    contents.append(f"[{text}]")
                else:
                    contents.append(text)

        # Chỉ lấy caption của figure
        elif child.tag_name == "figure" :
            ## If length > 200  --> not a caption, it's next description
            if len(text):
                if len(text) <= 150: # nếu mà len <= 150 --> add thêm [] xung quanh
                    contents.append(f"[{text}]")
                else:
                    contents.append(text)

        # Nếu mà là slide show thì nó giống figure
        elif child.tag_name == "div" and "item_slide_show" in child.get_attribute("class"):
            is_slide_show = True # slideshow
            if len(text):
                if len(text) <= 100:
                    contents.append(f"[{text}]")
                else:
                    contents.append(text)

        # Bỏ qua table bây giờ
        elif child.tag_name == "table": # Do nothing rightnow
            pass

    if is_slide_show:
        author = text

    # Nếu mà vẫn chưa thấy author thì tìm bằng tag
    if author == "Unknown":
        try:
            author = driver.find_element(by=By.XPATH, value="//*[contains(@class, 'author')]").text
        except:
            pass

    return {
        "url": article_url,
        "title": title,
        "description": description,
        "content": "\n".join(contents), # join các đoạn bằng \n
        "metadata": {
            "cat": main_cat,
            "subcat": sub_cat,
            "published_date": publish_date,
            "author": author
        }
    }


In [19]:
driver = webdriver.Chrome(options=chrome_options)

for cat, urls in url_data.items():

    print(f"Thu thập dữ liệu thể loại {cat} ..")
    count_crawled = 0
    cat_data = []
    for url in urls:
        try:
            cat_data.append(get_content_metadata(driver, url))
            count_crawled += 1
            if MAX_ARTICLES_PER_CAT and count_crawled >= MAX_ARTICLES_PER_CAT:
                break

        except (StaleElementReferenceException, NoSuchElementException) as e:
            print(f"Bug at url: {url}, with ElementException")
            driver.refresh()
            # Chúng ta tạm bỏ lỗi bây giờ
            continue

    name_file_cat = cat.lower().replace(" ", "-") + ".json"

    with open(os.path.join(DATA_FOLDER_OUTPUT, name_file_cat), "w", encoding="utf-8") as fOut:
        json.dump(cat_data, fOut, ensure_ascii=False, indent=4)

driver.close()

Thu thập dữ liệu thể loại Thể thao ..
Bug at url: https://vnexpress.net/toa-an-de-nghi-tuoc-huy-chuong-cua-vdv-my-tai-olympic-2024-4780175.html, with ElementException
Bug at url: https://vnexpress.net/guardiola-haaland-co-the-canh-tranh-ky-luc-cua-messi-ronaldo-4785361.html, with ElementException
Bug at url: https://vnexpress.net/vo-si-judo-141-kg-xach-co-ao-doi-thu-83-kg-4777440.html, with ElementException
Bug at url: https://vnexpress.net/chuoi-su-kien-ra-mat-giay-chay-bo-nike-pegasus-41-4781438.html, with ElementException
Bug at url: https://vnexpress.net/hon-2-000-nguoi-tham-gia-series-chay-bo-hanh-trinh-viet-nam-4783028.html, with ElementException
Bug at url: https://vnexpress.net/an-do-khieu-nai-vu-do-vat-bi-loai-khoi-olympic-vi-thua-0-15-kg-4778972.html, with ElementException
Bug at url: https://vnexpress.net/nha-vo-dich-olympic-he-lo-them-goc-khuat-ve-lang-vdv-o-paris-4782118.html, with ElementException
Bug at url: https://vnexpress.net/bayern-thang-nhoc-tran-ra-quan-bundesliga

In [46]:
# Xem 1 sample
cat_data[0]

{'url': 'https://vnexpress.net/suzuki-xl7-hybrid-ra-mat-cuoi-thang-8-tai-viet-nam-4778788.html',
 'title': 'Suzuki XL7 hybrid ra mắt cuối tháng 8 tại Việt Nam',
 'description': 'Phiên bản hybrid nhẹ của XL7 dự kiến trình làng vào 20/8, tiếp tục nhập khẩu Indonesia.',
 'content': 'Suzuki Việt Nam xác nhận sẽ ra mắt và bán XL7 hybrid 2024 vào 20/8 tới. Các đại lý đã nhận cọc phiên bản mới của Suzuki XL7 từ nhiều tháng qua.\nTheo các nhân viên bán hàng đại lý Suzuki, mẫu MPV 7 chỗ XL7 mới sẽ có thêm một số trang bị như kiểm soát hành trình, đèn pha tự động kèm tính năng đèn chờ dẫn đường. Thiết kế không thay đổi nhiều so với bản cũ.\nNâng cấp lớn nhất trên Suzuki XL7 2024 là sự xuất hiện của động cơ mild-hybrid, loại đã áp dụng trên Ertiga hybrid trước đó. Trên XL7 hybrid, động cơ 1,5 lít hút khí tự nhiên VVT có thêm một máy phát điện khởi động tích hợp (ISG) và một gói pin lithium-ion dung lượng nhỏ nằm dưới ghế lái phụ. Hệ thống cho công suất 103 mã lực và mô-men xoắn 138 Nm tại vòng tu

## Lưu dữ liệu

Nếu bạn chạy ở máy cá nhân thì không cần, nhưng nếu mà chạy ở Colab thì nên lưu dữ liệu vào trong Google Drive


In [ ]:
# For Google Colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Set to your folder
FOLDER_SAVED_GOOGLE_COLAB = "/content/drive/MyDrive/crawl-news/"

# Copy
!cp -r data $FOLDER_SAVED_GOOGLE_COLAB